# 02 – Data Cleaning, Transformation & Integration (V2)

**Research question Q1:** How do major events in Canton Zürich relate to daily POS spending across merchant categories compared with similar non-event days? (2019–2025)

In this notebook we prepare the data for the analysis in Notebook 03. We
1. clean the daily Zürich spending data for three merchant categories,
2. combine the 11 event days into 7 event editions,
3. create a window of ±28 days around every edition,
4. join the windows with the spending data and export three CSV files.

Main decisions:
* **One row per edition:** Züri Fäscht lasts three days but counts as one edition, not three separate events.
* **Day 0:** the Street Parade day, or the Saturday (middle day) of Züri Fäscht. Friday is then day −1 and Sunday day +1.
* **Attendance per edition only:** we do not use `estimated_daily_attendance`, because the sources do not support splitting attendance equally across days.
* **Spending values unchanged:** we do not rescale them. MCS divides all values by the same number, so percentage comparisons remain valid.

Each step is written as a function. The export at the end runs all steps again from the input files, so a file is only saved if every check passes.

## 2. Load datasets

The notebook must be run from the project folder (the folder that contains `data/`). The input files are only read, never changed.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SPENDING_FILE = Path("data/raw/ACQ_POS_Grossregion_NOGA.csv")
EVENTS_FILE = Path("data/processed/zurich_event_days.csv")
OUTPUT_DIR = Path("data/processed")

START_DATE = "2019-01-01"
END_DATE = "2025-12-31"
CATEGORIES = ["Accommodation", "Food and beverage services", "Transport services"]  # exact labels in the data
WINDOW_DAYS = 28


def load_inputs():
    for file in [SPENDING_FILE, EVENTS_FILE]:
        if not file.exists():
            raise FileNotFoundError(f"{file} not found - run the notebook from the project folder")
    # Region contains the code "9999", so both text columns are read as strings
    spending_raw = pd.read_csv(SPENDING_FILE, dtype={"Region": str, "Merchant category": str})
    # the event file was saved with a BOM (for Excel), utf-8-sig removes it
    events_raw = pd.read_csv(EVENTS_FILE, encoding="utf-8-sig")
    return spending_raw, events_raw


spending_raw, events_raw = load_inputs()
print("Spending:", spending_raw.shape)
print("Events:  ", events_raw.shape)

Spending: (156664, 5)
Events:   (11, 8)


## 3. Validate and clean spending data

We first check the full spending file:
* expected columns,
* no missing values,
* valid dates,
* positive numeric values,
* no duplicate records.

We then keep only Zürich, 2019–2025 and the three merchant categories. Every category must have one value for each of the 2,557 days, which gives 7,671 rows. Problems stop the notebook; duplicates are not averaged and gaps are not filled.

In [2]:
def clean_spending(spending_raw):
    expected_columns = ["Date", "Region", "Merchant category", "Scaled Value", "Scaled Number of Transactions"]
    assert list(spending_raw.columns) == expected_columns, f"Unexpected columns: {list(spending_raw.columns)}"
    assert spending_raw.notna().all().all(), "Spending data contain missing values"

    spending = spending_raw.rename(columns={
        "Date": "date",
        "Region": "region",
        "Merchant category": "merchant_category",
        "Scaled Value": "scaled_value",
        "Scaled Number of Transactions": "scaled_n_transactions",
    })
    spending["date"] = pd.to_datetime(spending["date"], format="%Y-%m-%d")  # fails on an invalid date

    for column in ["scaled_value", "scaled_n_transactions"]:
        assert pd.api.types.is_numeric_dtype(spending[column]), f"{column} is not numeric"
        assert np.isfinite(spending[column]).all(), f"{column} contains infinite values"
        assert (spending[column] > 0).all(), f"{column} contains zero or negative values"

    duplicates = spending.duplicated(["date", "region", "merchant_category"]).sum()
    assert duplicates == 0, f"{duplicates} duplicate date/region/category records"

    # keep Zürich, 2019-2025 and the three categories
    is_zurich = spending["region"] == "Zurich"
    in_period = spending["date"].between(START_DATE, END_DATE)
    in_categories = spending["merchant_category"].isin(CATEGORIES)
    spending = spending[is_zurich & in_period & in_categories]
    spending = spending.sort_values(["date", "merchant_category"]).reset_index(drop=True)

    # every category needs exactly one value per day
    all_days = pd.date_range(START_DATE, END_DATE)
    for category in CATEGORIES:
        days = spending.loc[spending["merchant_category"] == category, "date"]
        assert len(days) == len(all_days) and set(days) == set(all_days), f"{category}: dates missing or duplicated"

    assert len(spending) == 7671, f"Expected 7,671 rows, found {len(spending)}"
    return spending


spending = clean_spending(spending_raw)
print("Clean spending:", spending.shape)
print(spending.groupby("merchant_category")["date"].agg(["count", "min", "max"]))

Clean spending: (7671, 5)
                            count        min        max
merchant_category                                      
Accommodation                2557 2019-01-01 2025-12-31
Food and beverage services   2557 2019-01-01 2025-12-31
Transport services           2557 2019-01-01 2025-12-31


## 4. Validate and prepare event data

The event file has one row per event day. We check that, within each edition:
* the day numbers follow the dates (1, 2, 3),
* the days are consecutive,
* the year matches the date,
* attendance and sources are the same on every day.

We then combine the days into one row per edition. Day 0 (`event_reference_date`) is the middle day, which is the Saturday for both events. The `event_id` is built from the initials and the year, e.g. `SP_2019` or `ZF_2023`.

In [3]:
def make_event_id(name, year):
    initials = "".join(word[0] for word in name.split())   # "Street Parade" -> "SP"
    return f"{initials}_{year}"


def prepare_editions(events_raw):
    expected_columns = ["date", "event_name", "event_year", "edition_attendance", "estimated_daily_attendance",
                        "event_day_number", "date_source", "attendance_source"]
    assert list(events_raw.columns) == expected_columns, f"Unexpected columns: {list(events_raw.columns)}"
    assert events_raw.notna().all().all(), "Event data contain missing values"

    events = events_raw.copy()
    events["date"] = pd.to_datetime(events["date"], format="%Y-%m-%d")
    events = events.sort_values(["event_name", "event_year", "date"])
    assert (events["event_year"] == events["date"].dt.year).all(), "event_year does not match the event date"
    assert not events.duplicated(["event_name", "event_year", "date"]).any(), "Duplicate date within an edition"

    rows = []
    for (name, year), days in events.groupby(["event_name", "event_year"]):
        n_days = len(days)
        assert days["event_day_number"].tolist() == list(range(1, n_days + 1)), f"{name} {year}: wrong day numbers"
        assert (days["date"].diff().dropna() == pd.Timedelta(days=1)).all(), f"{name} {year}: days not consecutive"
        for column in ["edition_attendance", "date_source", "attendance_source"]:
            assert days[column].nunique() == 1, f"{name} {year}: different {column} values"

        start, end = days["date"].min(), days["date"].max()
        rows.append({
            "event_id": make_event_id(name, year),
            "event_name": name,
            "event_year": year,
            "event_start_date": start,
            "event_end_date": end,
            "event_reference_date": start + pd.Timedelta(days=(n_days - 1) // 2),   # middle day
            "event_duration_days": n_days,
            "edition_attendance": days["edition_attendance"].iloc[0],
            "date_source": days["date_source"].iloc[0],
            "attendance_source": days["attendance_source"].iloc[0],
        })
    editions = pd.DataFrame(rows).sort_values("event_start_date").reset_index(drop=True)

    # expected selection: 5 one-day Street Parades and 2 three-day Züri Fäscht editions, always centred on a Saturday
    durations = editions.groupby("event_name")["event_duration_days"].apply(list).to_dict()
    assert durations == {"Street Parade": [1] * 5, "Züri Fäscht": [3] * 2}, f"Unexpected editions: {durations}"
    assert (editions["event_reference_date"].dt.day_name() == "Saturday").all(), "Day 0 is not always a Saturday"
    return editions


editions = prepare_editions(events_raw)
editions

,event_id,event_name,event_year,event_start_date,event_end_date,event_reference_date,event_duration_days,edition_attendance,date_source,attendance_source
0,ZF_2019,Züri Fäscht,2019,2019-07-05,2019-07-07,2019-07-06,3,2500000,https://www.zhk.ch/de/wirtschaft-und-politik/m...,https://de.wikipedia.org/wiki/Z%C3%BCri_F%C3%A...
1,SP_2019,Street Parade,2019,2019-08-10,2019-08-10,2019-08-10,1,850000,https://www.stadt-zuerich.ch/misc/de/mitteilun...,https://www.streetparade.com/en/historie
2,SP_2022,Street Parade,2022,2022-08-13,2022-08-13,2022-08-13,1,900000,https://www.stadt-zuerich.ch/de/aktuell/medien...,https://www.streetparade.com/en/historie
3,ZF_2023,Züri Fäscht,2023,2023-07-07,2023-07-09,2023-07-08,3,2000000,https://www.freizeit.ch/kalender/zuri-fascht/,https://www.swissinfo.ch/eng/society/two-milli...
4,SP_2023,Street Parade,2023,2023-08-12,2023-08-12,2023-08-12,1,920000,https://polizei.news/2023/08/09/stadt-zuerich-...,https://www.streetparade.com/en/historie
5,SP_2024,Street Parade,2024,2024-08-10,2024-08-10,2024-08-10,1,920000,https://www.polizeinews.ch/2024/08/06/stadt-zu...,https://www.streetparade.com/en/historie
6,SP_2025,Street Parade,2025,2025-08-09,2025-08-09,2025-08-09,1,800000,https://polizeiticker.ch/artikel/street-parade...,https://www.streetparade.com/en/historie


## 5. Create event windows

For every edition we create the 57 days from day −28 to day +28. `is_event_day` marks the actual event days: day 0 for Street Parade and days −1, 0 and +1 for Züri Fäscht.

`is_event_day = False` only means "not an event day". Which of these days are suitable as a baseline is decided in Notebook 03.

In [4]:
def create_windows(editions):
    rows = []
    for _, edition in editions.iterrows():
        for relative_day in range(-WINDOW_DAYS, WINDOW_DAYS + 1):
            date = edition["event_reference_date"] + pd.Timedelta(days=relative_day)
            rows.append({
                "event_id": edition["event_id"],
                "date": date,
                "relative_day": relative_day,
                "is_event_day": edition["event_start_date"] <= date <= edition["event_end_date"],
            })
    windows = pd.DataFrame(rows)

    # check the event-day flags: 1-day event -> day 0, 3-day event -> days -1, 0, +1
    event_days = windows[windows["is_event_day"]]
    for _, edition in editions.iterrows():
        flagged = event_days.loc[event_days["event_id"] == edition["event_id"], "relative_day"].tolist()
        half = (edition["event_duration_days"] - 1) // 2
        assert flagged == list(range(-half, half + 1)), f"{edition['event_id']}: wrong event days {flagged}"
    assert len(event_days) == 11, f"Expected 11 event days, found {len(event_days)}"
    assert len(windows) == len(editions) * 57, "Every edition needs 57 window days"
    return windows


windows = create_windows(editions)
print("Window days:", len(windows))
print("Event days:", windows[windows["is_event_day"]].groupby("event_id")["relative_day"].apply(list).to_dict())
print("Calendar dates that appear in two windows:", windows["date"].duplicated().sum())

Window days:

 399
Event days: {'SP_2019': [0], 'SP_2022': [0], 'SP_2023': [0], 'SP_2024': [0], 'SP_2025': [0], 'ZF_2019': [-1, 0, 1], 'ZF_2023': [-1, 0, 1]}
Calendar dates that appear in two windows: 44


## 6. Integrate datasets

Each window day is repeated for the three merchant categories and then joined with the spending data on `date` and `merchant_category`.

`validate="many_to_one"` makes pandas stop if the spending data have more than one row for a date and category. If two windows share a calendar date, that date stays in both windows as separate rows. The result has one row per edition × day × category.

In [5]:
def integrate(windows, spending):
    categories = pd.DataFrame({"merchant_category": CATEGORIES})
    window_rows = windows.merge(categories, how="cross")       # every window day x 3 categories

    integrated = window_rows.merge(
        spending[["date", "merchant_category", "scaled_value", "scaled_n_transactions"]],
        on=["date", "merchant_category"],
        how="left",
        validate="many_to_one",
    )
    assert integrated["scaled_value"].notna().all(), "Some window days have no spending data"

    columns = ["event_id", "date", "relative_day", "is_event_day",
               "merchant_category", "scaled_value", "scaled_n_transactions"]
    return integrated[columns]


integrated = integrate(windows, spending)
print("Integrated:", integrated.shape)
integrated.head(6)

Integrated: (1197, 7)


,event_id,date,relative_day,is_event_day,merchant_category,scaled_value,scaled_n_transactions
0,ZF_2019,2019-06-08,-28,False,Accommodation,0.451871,0.208857
1,ZF_2019,2019-06-08,-28,False,Food and beverage services,1.439116,3.109406
2,ZF_2019,2019-06-08,-28,False,Transport services,0.093829,0.472707
3,ZF_2019,2019-06-09,-27,False,Accommodation,0.402302,0.173408
4,ZF_2019,2019-06-09,-27,False,Food and beverage services,1.094515,2.323974
5,ZF_2019,2019-06-09,-27,False,Transport services,0.077732,0.324009


## 7. Final checks and export

`check_outputs` tests the three tables once more before saving: size, columns, missing values, unique keys and three categories per window day.

The export cell does not reuse variables from the cells above. It loads the input files again and repeats every step. This means nothing is saved if any check fails, even if cells were run out of order or a variable was changed by hand.

In [6]:
def check_outputs(spending, editions, integrated):
    assert spending.shape == (7671, 5), f"Spending shape {spending.shape}"
    assert editions.shape == (7, 10), f"Editions shape {editions.shape}"
    assert integrated.shape == (1197, 7), f"Integrated shape {integrated.shape}"   # 7 editions x 57 days x 3 categories
    assert integrated.notna().all().all(), "Missing values in the integrated data"
    assert not integrated.duplicated(["event_id", "date", "merchant_category"]).any(), "Duplicate keys"
    categories_per_day = integrated.groupby(["event_id", "relative_day"])["merchant_category"].nunique()
    assert (categories_per_day == 3).all(), "Not every window day has three categories"
    assert integrated["is_event_day"].sum() == 11 * 3, "Expected 11 event days x 3 categories"

In [7]:
# Run the whole preparation again from the input files, check it, and only then save
spending_raw, events_raw = load_inputs()
spending = clean_spending(spending_raw)
editions = prepare_editions(events_raw)
windows = create_windows(editions)
integrated = integrate(windows, spending)
check_outputs(spending, editions, integrated)

outputs = {
    "q1_spending_zurich_clean.csv": spending,
    "q1_event_editions_clean.csv": editions,
    "q1_event_spending_integrated.csv": integrated,
}
for filename, table in outputs.items():
    # utf-8-sig so that "Züri Fäscht" displays correctly in Excel
    table.to_csv(OUTPUT_DIR / filename, index=False, date_format="%Y-%m-%d", encoding="utf-8-sig")
    print(f"Saved {OUTPUT_DIR / filename}: {table.shape[0]:,} rows x {table.shape[1]} columns")

Saved data/processed/q1_spending_zurich_clean.csv: 7,671 rows x 5 columns
Saved data/processed/q1_event_editions_clean.csv: 7 rows x 10 columns
Saved data/processed/q1_event_spending_integrated.csv: 1,197 rows x 7 columns


## 8. Summary

| File | Size | Key |
|---|---|---|
| `q1_spending_zurich_clean.csv` | 7,671 × 5 | `date`, `merchant_category` |
| `q1_event_editions_clean.csv` | 7 × 10 | `event_id` |
| `q1_event_spending_integrated.csv` | 1,197 × 7 | `event_id`, `date`, `merchant_category` |

**Integrated file**

| Column | Meaning |
|---|---|
| `event_id` | Edition, e.g. `SP_2019`; name, dates, attendance and sources are in the editions file |
| `date` | Calendar date (weekday, month etc. can be derived from it) |
| `relative_day` | Day relative to day 0, from −28 to +28 |
| `is_event_day` | True on the actual event days |
| `merchant_category` | Food and beverage services, Accommodation or Transport services |
| `scaled_value` | Scaled transaction value from MCS (not CHF) |
| `scaled_n_transactions` | Scaled number of transactions from MCS |

**Limitations**
* **Attendance:** `edition_attendance` is an organiser estimate for the whole edition, not per day. Züri Fäscht counts visits over three days, Street Parade a single day, so the two figures do not measure event size in the same way.
* **Coverage:** the data contain only seven editions of two summer events, not every major event in Zürich.
* **Baselines:** these are calculated in Notebook 03. Window days may still be affected by other events, public holidays (1 August lies in every window) or school holidays. The ZF and SP windows of 2019 and 2023 share 22 dates each.
* **Transport break:** Transport services shows a substantial structural break around January 25, 2021. The cause has not been confirmed. Possible explanations include changes in payment-data coverage, merchant classification, reporting methodology, or pandemic-related behavior. Comparisons across years should therefore be interpreted cautiously.